# IMPORTACIÓN DE LAS LIBRERÍAS Y DESCARGA DEL DATASET

In [ ]:
!pip install -q tcia_utils pydicom

In [ ]:
import pandas as pd
from tcia_utils import idc
import glob
import random
#from tcia_utils import nbia
import pydicom
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupShuffleSplit

In [ ]:
!wget -O dataset.tcia "https://www.cancerimagingarchive.net/wp-content/uploads/The-Chinese-Mammography-Database.tcia"

In [ ]:
!wget -O CMMD_c_r.xlsx "https://www.cancerimagingarchive.net/wp-content/uploads/CMMD_clinicaldata_revision.xlsx"

In [ ]:
df = idc.downloadSeries("dataset.tcia", input_type="manifest", format="df")

# EXPLORACIÓN DE LOS METADATOS

In [ ]:
df_clinico = pd.read_excel("CMMD_c_r.xlsx")

In [ ]:
display(df_clinico.head())

In [ ]:
print(f'--- Balanceo de clases --- \n {df_clinico['classification'].value_counts()} \n\n--- Pacientes únicos --- {df_clinico['ID1'].nunique()} \n\n--- Valores nulos --- {df_clinico[['ID1', 'classification']].isnull().sum()}')

**CASOS MALIGNOS**: 1316

**CASOS BENIGNOS**: 556

**VALORES NULOS**: 0

**PACIENTES**: 1775

Desbalanceo de clases con tendencia a la clase *maligno*

In [ ]:
from numpy.random import randint
from numpy.random.mtrand import random_integers
rutas_imagenes = glob.glob("**/*.dcm", recursive=True)

if rutas_imagenes:

    dicom_path = rutas_imagenes[randint(0,5202)]
    dicom_file = pydicom.dcmread(dicom_path)


    matriz_pixeles = dicom_file.pixel_array

    plt.figure(figsize=(10,10))
    plt.imshow(matriz_pixeles, cmap=plt.cm.bone)
    plt.title(f'Paciente {dicom_file.PatientID}')
    plt.axis('off')
    plt.show()

    print(f'Se encontraron {len(rutas_imagenes)} radiografías descargadas')
    print(f'ID paciente: {dicom_file.PatientID}\n\nResolución: {dicom_file.Rows} x {dicom_file.Columns}')
else:
    print('No se encontraron radiografías')

In [ ]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

generador_division = gss.split(df_clinico, groups=df_clinico['ID1'])
train_idx, test_idx = next(generador_division)

df_train = df_clinico.iloc[train_idx]
df_test = df_clinico.iloc[test_idx]

print(f'Pacientes de entrenamiento: {df_train["ID1"].nunique()}')
print(f'Pacientes de prueba: {df_test["ID1"].nunique()}')

#--- VERIFICACIÓN ---#
pacientes_train = set(df_train['ID1'])
pacientes_test = set(df_test['ID1'])
interseccion = pacientes_train.intersection(pacientes_test)

print(f'Pacientes en ambos conjuntos: {len(interseccion)}')

In [ ]:
# --- CÓDIGO GENERADO CON GEMINI --- #

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import numpy as np
import pydicom
import glob

# 1. Definir el mapeo de etiquetas (0 = Benigno, 1 = Maligno)
etiquetas_map = {'Benign': 0, 'Malignant': 1}
# Hacemos una copia profunda para que Pandas sepa que son tablas independientes y quite la alerta
df_train = df_train.copy()
df_test = df_test.copy()

df_train.loc[:, 'label'] = df_train['classification'].map(etiquetas_map)
df_test.loc[:, 'label'] = df_test['classification'].map(etiquetas_map)
# 2. Pipeline de preprocesamiento (Resize, Tensor, Normalization)
# Se define 224x224 para la entrada de ResNet18
transformaciones = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

# 3. Clase personalizada Dataset para leer imágenes DICOM bajo demanda
class CMMD_Dataset(Dataset):
    def __init__(self, dataframe, image_paths, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.image_paths = image_paths
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        # Obtener ID del paciente y etiqueta
        paciente_id = self.df.loc[idx, 'ID1']
        label = self.df.loc[idx, 'label']

        # Buscar el archivo .dcm correspondiente al paciente
        ruta_img = next((ruta for ruta in self.image_paths if paciente_id in ruta), None)

        if ruta_img is None:
            # Fallback en caso de imagen no encontrada
            img_tensor = torch.zeros((1, 224, 224))
            return img_tensor, torch.tensor(label, dtype=torch.long)

        # Leer y procesar píxeles DICOM
        dicom_data = pydicom.dcmread(ruta_img)
        img_array = dicom_data.pixel_array

        # Normalizar escala de grises al rango 0-255
        img_array = img_array - np.min(img_array)
        img_array = (img_array / np.max(img_array) * 255).astype(np.uint8)

        if self.transform:
            img_tensor = self.transform(img_array)

        return img_tensor, torch.tensor(label, dtype=torch.long)

# 4. Obtener lista de rutas y crear DataLoaders
todas_las_rutas = glob.glob("**/*.dcm", recursive=True)

train_dataset = CMMD_Dataset(df_train, todas_las_rutas, transform=transformaciones)
test_dataset = CMMD_Dataset(df_test, todas_las_rutas, transform=transformaciones)

# Ajustar batch_size según la VRAM (32 es un buen punto de partida para Google Colab T4)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# Verificación rápida del primer lote para la demostración en vivo
imagenes_batch, etiquetas_batch = next(iter(train_loader))
print(f"Dimensiones del lote de imágenes: {imagenes_batch.shape}")
print(f"Dimensiones del lote de etiquetas: {etiquetas_batch.shape}")

In [ ]:
# --- PROCESAMIENTO DE 5 LOTES. PENSADO PARA PRUEBA EN VIVO --- #

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models

# 1. Configurar el "Cerebro" (Usar la tarjeta gráfica GPU de Colab si está disponible)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"El modelo se entrenará usando: {device}\n")

# 2. Contratar al experto (Transfer Learning con ResNet18)
# pretrained=True descarga el "cerebro" que ya sabe detectar formas y texturas
modelo = models.resnet18(pretrained=True)

# 2.1 Ponerle las gafas de rayos X: Modificar la primera capa para aceptar 1 canal (Blanco y negro) en vez de 3 (RGB)
modelo.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)

# 2.2 Limitar sus respuestas: Modificar la última capa para que solo arroje 2 salidas (Benigno o Maligno)
num_ftrs = modelo.fc.in_features
modelo.fc = nn.Linear(num_ftrs, 2)

# Enviar el modelo a la tarjeta gráfica
modelo = modelo.to(device)

# 3. Definir cómo va a aprender
criterio = nn.CrossEntropyLoss() # El juez que le dice qué tan equivocado está
optimizador = optim.Adam(modelo.parameters(), lr=0.001) # El mecanismo que ajusta sus neuronas

# 4. LA DEMOSTRACIÓN EN VIVO: Entrenamiento Preliminar
print("--- INICIANDO ENTRENAMIENTO PRELIMINAR (DEMO) ---")
modelo.train()

# Solo pasaremos 5 "cajas" (lotes) por la cinta transportadora para la demostración
lotes_a_mostrar = 5

for i, (inputs, labels) in enumerate(train_loader):
    if i >= lotes_a_mostrar:
        break

    # Enviar las imágenes y las respuestas correctas a la GPU
    inputs, labels = inputs.to(device), labels.to(device)

    # 1. Limpiar su memoria a corto plazo
    optimizador.zero_grad()

    # 2. Intentar adivinar (Forward pass)
    outputs = modelo(inputs)
    loss = criterio(outputs, labels) # Calcular el error

    # 3. Aprender de su error (Backward pass)
    loss.backward()
    optimizador.step()

    # Mostrar el resultado en pantalla
    print(f"Lote {i+1}/{lotes_a_mostrar} procesado con éxito | Error matemático (Loss): {loss.item():.4f}")

print("\n¡Entrenamiento preliminar completado! El modelo no explotó y está ajustando sus pesos.")

# LÍNEA BASE -- MODELO ENTRENADO

In [ ]:
import os
import time
import random
import numpy as np
import pandas as pd
import pydicom
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    confusion_matrix, classification_report, roc_auc_score,
    recall_score, precision_score, f1_score
)

# 1. Fijar semillas para reproducibilidad
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# 2. Indexar TODAS las imágenes DICOM vinculando PatientID y Laterality (L/R)
registros_imagenes = []
etiquetas_map = {'Benign': 0, 'Malignant': 1}
df_clinico['label'] = df_clinico['classification'].map(etiquetas_map)

# Crear diccionario rápido de etiquetas por (ID1, LeftRight)
mapa_clinico = {
    (row['ID1'], row['LeftRight']): row['label']
    for _, row in df_clinico.iterrows()
}

for ruta in todas_las_rutas:
    ds = pydicom.dcmread(ruta, stop_before_pixels=True) # Lectura rápida solo de metadatos
    pid = getattr(ds, 'PatientID', None)
    lat = getattr(ds, 'ImageLaterality', None) or getattr(ds, 'Laterality', None)

    if (pid, lat) in mapa_clinico:
        registros_imagenes.append({
            'ruta': ruta,
            'ID1': pid,
            'LeftRight': lat,
            'label': mapa_clinico[(pid, lat)]
        })

df_imagenes = pd.DataFrame(registros_imagenes)
print(f"Total imágenes emparejadas correctamente: {len(df_imagenes)}")

# 3. División agrupada por paciente: 70% Train, 15% Val, 15% Test
gss_test = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=SEED)
train_val_idx, test_idx = next(gss_test.split(df_imagenes, groups=df_imagenes['ID1']))
df_train_val = df_imagenes.iloc[train_val_idx].reset_index(drop=True)
df_test = df_imagenes.iloc[test_idx].reset_index(drop=True)

gss_val = GroupShuffleSplit(n_splits=1, test_size=0.1765, random_state=SEED) # 0.1765 * 0.85 ≈ 0.15
train_idx, val_idx = next(gss_val.split(df_train_val, groups=df_train_val['ID1']))
df_train = df_train_val.iloc[train_idx].reset_index(drop=True)
df_val = df_train_val.iloc[val_idx].reset_index(drop=True)

# Verificación estricta de cero fuga de pacientes
assert len(set(df_train['ID1']) & set(df_val['ID1'])) == 0, "¡Fuga entre Train y Val!"
assert len(set(df_train['ID1']) & set(df_test['ID1'])) == 0, "¡Fuga entre Train y Test!"
assert len(set(df_val['ID1']) & set(df_test['ID1'])) == 0, "¡Fuga entre Val y Test!"

print(f"Imágenes -> Train: {len(df_train)} | Val: {len(df_val)} | Test: {len(df_test)}")
print(f"Pacientes -> Train: {df_train['ID1'].nunique()} | Val: {df_val['ID1'].nunique()} | Test: {df_test['ID1'].nunique()}")

In [ ]:
# Transformaciones de Línea Base (3 canales para conservar pesos ImageNet intactos)
transform_base = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

class CMMD_Dataset_Base(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        ruta_img = self.df.loc[idx, 'ruta']
        label = self.df.loc[idx, 'label']

        dicom_data = pydicom.dcmread(ruta_img)
        img_array = dicom_data.pixel_array.astype(np.float32)

        # Normalización segura a [0, 255]
        min_val, max_val = np.min(img_array), np.max(img_array)
        if max_val > min_val:
            img_array = ((img_array - min_val) / (max_val - min_val) * 255.0).astype(np.uint8)
        else:
            img_array = np.zeros_like(img_array, dtype=np.uint8)

        if self.transform:
            img_tensor = self.transform(img_array)

        return img_tensor, torch.tensor(label, dtype=torch.long)

train_loader = DataLoader(CMMD_Dataset_Base(df_train, transform_base), batch_size=32, shuffle=True, num_workers=2)
val_loader   = DataLoader(CMMD_Dataset_Base(df_val, transform_base), batch_size=32, shuffle=False, num_workers=2)
test_loader  = DataLoader(CMMD_Dataset_Base(df_test, transform_base), batch_size=32, shuffle=False, num_workers=2)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Modelo Línea Base: ResNet18 preentrenado
modelo_base = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
modelo_base.fc = nn.Linear(modelo_base.fc.in_features, 2)
modelo_base = modelo_base.to(device)

criterio = nn.CrossEntropyLoss()
optimizador = optim.Adam(modelo_base.parameters(), lr=1e-4)
EPOCHS = 10

inicio_entrenamiento = time.time()
historial = {'train_loss': [], 'val_loss': [], 'val_recall': [], 'val_auc': []}

for epoch in range(EPOCHS):
    # --- ENTRENAMIENTO ---
    modelo_base.train()
    running_loss = 0.0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizador.zero_grad()
        outputs = modelo_base(inputs)
        loss = criterio(outputs, labels)
        loss.backward()
        optimizador.step()
        running_loss += loss.item() * inputs.size(0)

    train_loss = running_loss / len(train_loader.dataset)

    # --- VALIDACIÓN ---
    modelo_base.eval()
    val_loss = 0.0
    y_true, y_probs = [], []
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = modelo_base(inputs)
            loss = criterio(outputs, labels)
            val_loss += loss.item() * inputs.size(0)

            probs = torch.softmax(outputs, dim=1)[:, 1]
            y_true.extend(labels.cpu().numpy())
            y_probs.extend(probs.cpu().numpy())

    val_loss /= len(val_loader.dataset)
    y_pred = (np.array(y_probs) >= 0.5).astype(int)

    val_recall = recall_score(y_true, y_pred) # Sensibilidad clase Malignant
    val_auc = roc_auc_score(y_true, y_probs)

    historial['train_loss'].append(train_loss)
    historial['val_loss'].append(val_loss)
    historial['val_recall'].append(val_recall)
    historial['val_auc'].append(val_auc)

    print(f"Época {epoch+1:02d}/{EPOCHS:02d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Recall (Malignant): {val_recall:.4f} | Val AUC: {val_auc:.4f}")

tiempo_total = time.time() - inicio_entrenamiento
print(f"\nTiempo total de entrenamiento: {tiempo_total:.2f} segundos")

# Guardar pesos de la línea base para comparaciones futuras
torch.save(modelo_base.state_dict(), "resnet18_linea_base.pth")

# EXPERIMENTO 1


In [ ]:
# 1. Reiniciar semillas para reproducibilidad exacta respecto a la Línea Base
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 2. Instanciar modelo limpio: ResNet18 preentrenado (misma arquitectura que Línea Base)
modelo_exp1 = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
modelo_exp1.fc = nn.Linear(modelo_exp1.fc.in_features, 2)
modelo_exp1 = modelo_exp1.to(device)

# 3. ÚNICO CAMBIO CONTROLADO: Calcular pesos de clase usando SOLO df_train
conteos_train = df_train['label'].value_counts().sort_index()  # 0: Benign, 1: Malignant
total_train = len(df_train)
pesos_clase = torch.tensor([
    total_train / (2.0 * conteos_train[0]),
    total_train / (2.0 * conteos_train[1])
], dtype=torch.float).to(device)

print(f"Pesos aplicados [Benign (0), Malignant (1)]: {pesos_clase.cpu().numpy()}")

criterio_exp1 = nn.CrossEntropyLoss(weight=pesos_clase)
optimizador_exp1 = optim.Adam(modelo_exp1.parameters(), lr=1e-4)
EPOCHS = 10

inicio_entrenamiento = time.time()
historial_exp1 = {'train_loss': [], 'val_loss': [], 'val_recall': [], 'val_auc': []}

for epoch in range(EPOCHS):
    # --- ENTRENAMIENTO ---
    modelo_exp1.train()
    running_loss = 0.0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizador_exp1.zero_grad()
        outputs = modelo_exp1(inputs)
        loss = criterio_exp1(outputs, labels)
        loss.backward()
        optimizador_exp1.step()
        running_loss += loss.item() * inputs.size(0)

    train_loss = running_loss / len(train_loader.dataset)

    # --- VALIDACIÓN ---
    modelo_exp1.eval()
    val_loss = 0.0
    y_true_exp1, y_probs_exp1 = [], []
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = modelo_exp1(inputs)
            loss = criterio_exp1(outputs, labels)
            val_loss += loss.item() * inputs.size(0)

            probs = torch.softmax(outputs, dim=1)[:, 1]
            y_true_exp1.extend(labels.cpu().numpy())
            y_probs_exp1.extend(probs.cpu().numpy())

    val_loss /= len(val_loader.dataset)
    y_pred_exp1 = (np.array(y_probs_exp1) >= 0.5).astype(int)

    val_recall = recall_score(y_true_exp1, y_pred_exp1)
    val_auc = roc_auc_score(y_true_exp1, y_probs_exp1)

    historial_exp1['train_loss'].append(train_loss)
    historial_exp1['val_loss'].append(val_loss)
    historial_exp1['val_recall'].append(val_recall)
    historial_exp1['val_auc'].append(val_auc)

    print(f"Época {epoch+1:02d}/{EPOCHS:02d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Recall (Malignant): {val_recall:.4f} | Val AUC: {val_auc:.4f}")

tiempo_total_exp1 = time.time() - inicio_entrenamiento
print(f"\nTiempo total de entrenamiento Exp. 1: {tiempo_total_exp1:.2f} segundos")

# Guardar pesos del Experimento 1
torch.save(modelo_exp1.state_dict(), "resnet18_exp1_weights.pth")

# --- REPORTE DE MÉTRICAS EXPERIMENTO 1 (VALIDACIÓN, UMBRAL 0.5) ---
cm_exp1 = confusion_matrix(y_true_exp1, y_pred_exp1)
tn, fp, fn, tp = cm_exp1.ravel()
especificidad_exp1 = tn / (tn + fp)

print("\n--- MÉTRICAS EXPERIMENTO 1 (CONJUNTO DE VALIDACIÓN, UMBRAL 0.5) ---")
print(f"Matriz de Confusión:\n{cm_exp1}")
print(f"Sensibilidad / Recall (Malignant): {recall_score(y_true_exp1, y_pred_exp1):.4f}")
print(f"Especificidad (Benign):            {especificidad_exp1:.4f}")
print(f"Precisión (Malignant):             {precision_score(y_true_exp1, y_pred_exp1):.4f}")
print(f"F1-Score (Malignant):              {f1_score(y_true_exp1, y_pred_exp1):.4f}")
print(f"ROC-AUC:                           {roc_auc_score(y_true_exp1, y_probs_exp1):.4f}")

# EXPERIMENTO 2

In [ ]:
# 1. Reiniciar semillas para garantizar las mismas condiciones iniciales
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 2. ÚNICO CAMBIO RESPECTO A LÍNEA BASE: Sin pesos preentrenados (weights=None)
modelo_exp2 = models.resnet18(weights=None)
modelo_exp2.fc = nn.Linear(modelo_exp2.fc.in_features, 2)
modelo_exp2 = modelo_exp2.to(device)

# Se mantienen exactamente los mismos hiperparámetros de la Línea Base
criterio_exp2 = nn.CrossEntropyLoss()
optimizador_exp2 = optim.Adam(modelo_exp2.parameters(), lr=1e-4)
EPOCHS = 10

inicio_entrenamiento = time.time()
historial_exp2 = {
    'train_loss': [], 'val_loss': [],
    'val_recall': [], 'val_f1': [], 'val_auc': []
}

for epoch in range(EPOCHS):
    # --- ENTRENAMIENTO ---
    modelo_exp2.train()
    running_loss = 0.0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizador_exp2.zero_grad()
        outputs = modelo_exp2(inputs)
        loss = criterio_exp2(outputs, labels)
        loss.backward()
        optimizador_exp2.step()
        running_loss += loss.item() * inputs.size(0)

    train_loss = running_loss / len(train_loader.dataset)

    # --- VALIDACIÓN ---
    modelo_exp2.eval()
    val_loss = 0.0
    y_true_exp2, y_probs_exp2 = [], []
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = modelo_exp2(inputs)
            loss = criterio_exp2(outputs, labels)
            val_loss += loss.item() * inputs.size(0)

            probs = torch.softmax(outputs, dim=1)[:, 1]
            y_true_exp2.extend(labels.cpu().numpy())
            y_probs_exp2.extend(probs.cpu().numpy())

    val_loss /= len(val_loader.dataset)
    y_pred_exp2 = (np.array(y_probs_exp2) >= 0.5).astype(int)

    val_recall = recall_score(y_true_exp2, y_pred_exp2)
    val_f1 = f1_score(y_true_exp2, y_pred_exp2)
    val_auc = roc_auc_score(y_true_exp2, y_probs_exp2)

    historial_exp2['train_loss'].append(train_loss)
    historial_exp2['val_loss'].append(val_loss)
    historial_exp2['val_recall'].append(val_recall)
    historial_exp2['val_f1'].append(val_f1)
    historial_exp2['val_auc'].append(val_auc)

    print(f"Época {epoch+1:02d}/{EPOCHS:02d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Recall: {val_recall:.4f} | Val F1: {val_f1:.4f} | Val AUC: {val_auc:.4f}")

tiempo_total_exp2 = time.time() - inicio_entrenamiento
print(f"\nTiempo total de entrenamiento Exp. 2 (From Scratch): {tiempo_total_exp2:.2f} segundos")

# Guardar pesos del Experimento 2
torch.save(modelo_exp2.state_dict(), "resnet18_exp2_scratch.pth")

# --- REPORTE DE MÉTRICAS EXPERIMENTO 2 (VALIDACIÓN, UMBRAL 0.5) ---
cm_exp2 = confusion_matrix(y_true_exp2, y_pred_exp2)
tn, fp, fn, tp = cm_exp2.ravel()
especificidad_exp2 = tn / (tn + fp)

print("\n--- MÉTRICAS EXPERIMENTO 2: FROM SCRATCH (VALIDACIÓN, UMBRAL 0.5) ---")
print(f"Matriz de Confusión:\n{cm_exp2}")
print(f"Sensibilidad / Recall (Malignant): {recall_score(y_true_exp2, y_pred_exp2):.4f}")
print(f"Especificidad (Benign):            {especificidad_exp2:.4f}")
print(f"Precisión (Malignant):             {precision_score(y_true_exp2, y_pred_exp2):.4f}")
print(f"F1-Score (Malignant):              {f1_score(y_true_exp2, y_pred_exp2):.4f}")
print(f"ROC-AUC:                           {roc_auc_score(y_true_exp2, y_probs_exp2):.4f}")

# EXPERIMENTO 3

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torchvision import models
from sklearn.metrics import (
    confusion_matrix, recall_score, precision_score,
    f1_score, roc_auc_score, roc_curve
)
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1. Cargar el mejor modelo preentrenado (cambia la ruta a "resnet18_exp1_weights.pth" si Exp. 1 superó a la Línea Base)
RUTA_MEJOR_MODELO = "resnet18_exp1_weights.pth"  # o "resnet18_linea_base.pth"

mejor_modelo = models.resnet18(weights=None)
mejor_modelo.fc = nn.Linear(mejor_modelo.fc.in_features, 2)
mejor_modelo.load_state_dict(torch.load(RUTA_MEJOR_MODELO, map_location=device))
mejor_modelo = mejor_modelo.to(device)
mejor_modelo.eval()

# 2. Obtener puntuaciones (scores) únicamente sobre el conjunto de VALIDACIÓN
y_true_val, y_scores_val = [], []

with torch.no_grad():
    for inputs, labels in val_loader:
        inputs = inputs.to(device)
        outputs = mejor_modelo(inputs)
        probs = torch.softmax(outputs, dim=1)[:, 1]  # Puntuación para la clase Malignant (1)
        y_true_val.extend(labels.numpy())
        y_scores_val.extend(probs.cpu().numpy())

y_true_val = np.array(y_true_val)
y_scores_val = np.array(y_scores_val)

# 3. Evaluar distintos umbrales de decisión (Contraste Hipótesis 2: 0.5 vs 0.3)
umbrales = [0.60, 0.50, 0.40, 0.35, 0.30, 0.25]
resultados_umbrales = []

for tau in umbrales:
    y_pred_tau = (y_scores_val >= tau).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true_val, y_pred_tau).ravel()

    sensibilidad = recall_score(y_true_val, y_pred_tau)          # Recall Malignant
    especificidad = tn / (tn + fp) if (tn + fp) > 0 else 0.0     # Recall Benign
    precision = precision_score(y_true_val, y_pred_tau, zero_division=0)
    f1 = f1_score(y_true_val, y_pred_tau, zero_division=0)

    resultados_umbrales.append({
        'Umbral (τ)': tau,
        'Recall / Sensibilidad': round(sensibilidad, 4),
        'Especificidad': round(especificidad, 4),
        'Precisión': round(precision, 4),
        'F1-Score': round(f1, 4),
        'Falsos Negativos (FN)': fn,
        'Falsos Positivos (FP)': fp,
        'Verdaderos Positivos (TP)': tp,
        'Verdaderos Negativos (TN)': tn
    })

df_umbrales = pd.DataFrame(resultados_umbrales)
print(f"ROC-AUC en Validación (independiente del umbral): {roc_auc_score(y_true_val, y_scores_val):.4f}\n")
display(df_umbrales)

# 4. Gráfica de compromiso Sensibilidad vs Especificidad y Curva ROC
fpr, tpr, _ = roc_curve(y_true_val, y_scores_val)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Curva ROC
axes[0].plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC (AUC = {roc_auc_score(y_true_val, y_scores_val):.4f})')
axes[0].plot([0, 1], [0, 1], color='navy', lw=1, linestyle='--')
axes[0].set_title('Curva ROC - Conjunto de Validación')
axes[0].set_xlabel('Tasa de Falsos Positivos (1 - Especificidad)')
axes[0].set_ylabel('Sensibilidad / Recall (Malignant)')
axes[0].legend(loc='lower right')
axes[0].grid(alpha=0.3)

# Trade-off por umbral
axes[1].plot(df_umbrales['Umbral (τ)'], df_umbrales['Recall / Sensibilidad'], marker='o', label='Sensibilidad (Recall Malignant)')
axes[1].plot(df_umbrales['Umbral (τ)'], df_umbrales['Especificidad'], marker='s', label='Especificidad (Benign)')
axes[1].plot(df_umbrales['Umbral (τ)'], df_umbrales['F1-Score'], marker='^', linestyle='--', label='F1-Score')
axes[1].set_title('Efecto del Umbral de Decisión (Validación)')
axes[1].set_xlabel('Umbral (τ)')
axes[1].set_ylabel('Valor de la métrica')
axes[1].invert_xaxis()  # Muestra de mayor a menor umbral (0.60 -> 0.25)
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

# EXTRAER VISUALIZACIÓN DE FALSOS NEGATIVOS Y FALSOS POSITIVOS (EN VALIDACIÓN)

In [ ]:
# 1. Definir el modelo y umbral seleccionados en Validación
RUTA_MODELO_FINAL = "resnet18_linea_base.pth" # Cambien a "resnet18_exp1_weights.pth" si prefieren el de Exp. 1
UMBRAL_OPTIMO = 0.30

modelo_final = models.resnet18(weights=None)
modelo_final.fc = nn.Linear(modelo_final.fc.in_features, 2)
modelo_final.load_state_dict(torch.load(RUTA_MODELO_FINAL, map_location=device))
modelo_final = modelo_final.to(device)
modelo_final.eval()

# 2. Obtener predicciones detalladas en Validación vinculadas a df_val
probs_val = []
with torch.no_grad():
    for inputs, _ in val_loader:
        inputs = inputs.to(device)
        outputs = modelo_final(inputs)
        probs = torch.softmax(outputs, dim=1)[:, 1]
        probs_val.extend(probs.cpu().numpy())

df_analisis_val = df_val.copy()
df_analisis_val['score_malignant'] = probs_val
df_analisis_val['pred'] = (df_analisis_val['score_malignant'] >= UMBRAL_OPTIMO).astype(int)

# Cruzar con metadatos clínicos originales (Age, abnormality) para enriquecer el análisis
meta_extra = df_clinico[['ID1', 'LeftRight', 'Age', 'abnormality']].drop_duplicates()
df_analisis_val = df_analisis_val.merge(meta_extra, on=['ID1', 'LeftRight'], how='left')

# 3. Filtrar Falsos Negativos (Real=1, Pred=0) y Falsos Positivos (Real=0, Pred=1)
falsos_negativos = df_analisis_val[(df_analisis_val['label'] == 1) & (df_analisis_val['pred'] == 0)]
falsos_positivos = df_analisis_val[(df_analisis_val['label'] == 0) & (df_analisis_val['pred'] == 1)]

print(f"Total Falsos Negativos (FN) con τ={UMBRAL_OPTIMO}: {len(falsos_negativos)}")
print("Distribución por tipo de hallazgo en FN:\n", falsos_negativos['abnormality'].value_counts(), "\n")

print(f"Total Falsos Positivos (FP) con τ={UMBRAL_OPTIMO}: {len(falsos_positivos)}")
print("Distribución por tipo de hallazgo en FP:\n", falsos_positivos['abnormality'].value_counts(), "\n")

# 4. Visualizar 3 ejemplos reales de Falsos Negativos y 3 de Falsos Positivos
def mostrar_ejemplos_error(df_errores, titulo, n=3):
    muestra = df_errores.head(n)
    fig, axes = plt.subplots(1, len(muestra), figsize=(15, 5))
    if len(muestra) == 1:
        axes = [axes]
    for ax, (_, row) in zip(axes, muestra.iterrows()):
        ds = pydicom.dcmread(row['ruta'])
        ax.imshow(ds.pixel_array, cmap=plt.cm.bone)
        ax.set_title(
            f"ID: {row['ID1']} ({row['LeftRight']}) | Edad: {row['Age']}\n"
            f"Hallazgo: {row['abnormality']} | Score: {row['score_malignant']:.3f}"
        )
        ax.axis('off')
    plt.suptitle(titulo, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

mostrar_ejemplos_error(falsos_negativos, f"Ejemplos de FALSOS NEGATIVOS (Real: Malignant, Pred: Benign | τ={UMBRAL_OPTIMO})")
mostrar_ejemplos_error(falsos_positivos, f"Ejemplos de FALSOS POSITIVOS (Real: Benign, Pred: Malignant | τ={UMBRAL_OPTIMO})")

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torchvision import models
from sklearn.metrics import (
    confusion_matrix, recall_score, precision_score,
    f1_score, roc_auc_score, classification_report
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1. Cargar el modelo ganador (Línea Base)
RUTA_MODELO_GANADOR = "resnet18_linea_base.pth"
modelo_ganador = models.resnet18(weights=None)
modelo_ganador.fc = nn.Linear(modelo_ganador.fc.in_features, 2)
modelo_ganador.load_state_dict(torch.load(RUTA_MODELO_GANADOR, map_location=device))
modelo_ganador = modelo_ganador.to(device)
modelo_ganador.eval()

# 2. Revisar brevemente los umbrales de la Línea Base en VALIDACIÓN para fijar τ*
y_true_val, y_scores_val = [], []
with torch.no_grad():
    for inputs, labels in val_loader:
        inputs = inputs.to(device)
        probs = torch.softmax(modelo_ganador(inputs), dim=1)[:, 1]
        y_true_val.extend(labels.numpy())
        y_scores_val.extend(probs.cpu().numpy())

y_true_val = np.array(y_true_val)
y_scores_val = np.array(y_scores_val)

print("=== SELECCIÓN DE UMBRAL EN VALIDACIÓN (LÍNEA BASE) ===")
for tau in [0.50, 0.40, 0.35, 0.30]:
    preds = (y_scores_val >= tau).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true_val, preds).ravel()
    print(f"τ = {tau:.2f} | Recall: {recall_score(y_true_val, preds):.4f} | Especif.: {tn/(tn+fp):.4f} | F1: {f1_score(y_true_val, preds):.4f} | FN: {fn} | FP: {fp}")

# Fijar umbral seleccionado (0.30 prioritario para reducir Falsos Negativos según Hipótesis 2)
UMBRAL_FINAL = 0.30

# 3. Análisis cuantitativo de errores en Validación (sin necesitar ojo médico)
df_err_val = df_val.copy()
df_err_val['score'] = y_scores_val
df_err_val['pred'] = (y_scores_val >= UMBRAL_FINAL).astype(int)

meta_extra = df_clinico[['ID1', 'LeftRight', 'Age', 'abnormality']].drop_duplicates()
df_err_val = df_err_val.merge(meta_extra, on=['ID1', 'LeftRight'], how='left')

fn_val = df_err_val[(df_err_val['label'] == 1) & (df_err_val['pred'] == 0)]
fp_val = df_err_val[(df_err_val['label'] == 0) & (df_err_val['pred'] == 1)]

print(f"\n=== ANÁLISIS DE ERRORES POR METADATOS (VALIDACIÓN, τ={UMBRAL_FINAL}) ===")
print(f"Falsos Negativos ({len(fn_val)} casos) por tipo de hallazgo:")
print(fn_val['abnormality'].value_counts(normalize=True).mul(100).round(2).astype(str) + '%')
print(f"Edad promedio en FN: {fn_val['Age'].mean():.1f} años\n")

print(f"Falsos Positivos ({len(fp_val)} casos) por tipo de hallazgo:")
print(fp_val['abnormality'].value_counts(normalize=True).mul(100).round(2).astype(str) + '%')
print(f"Edad promedio en FP: {fp_val['Age'].mean():.1f} años\n")

# ==============================================================================
# 4. EXAMEN FINAL: EVALUACIÓN ÚNICA EN EL CONJUNTO DE PRUEBA RESERVADO (TEST)
# ==============================================================================
y_true_test, y_scores_test = [], []
with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(device)
        probs = torch.softmax(modelo_ganador(inputs), dim=1)[:, 1]
        y_true_test.extend(labels.numpy())
        y_scores_test.extend(probs.cpu().numpy())

y_true_test = np.array(y_true_test)
y_scores_test = np.array(y_scores_test)
y_pred_test = (y_scores_test >= UMBRAL_FINAL).astype(int)

cm_test = confusion_matrix(y_true_test, y_pred_test)
tn_t, fp_t, fn_t, tp_t = cm_test.ravel()

print("=================================================================")
print(f"   RESULTADOS FINALES EN CONJUNTO DE PRUEBA (TEST) | τ = {UMBRAL_FINAL}")
print("=================================================================")
print(f"Matriz de Confusión:\n{cm_test}")
print(f"Verdaderos Positivos (TP): {tp_t} | Falsos Negativos (FN): {fn_t}")
print(f"Verdaderos Negativos (TN): {tn_t} | Falsos Positivos (FP): {fp_t}")
print("-" * 65)
print(f"Sensibilidad / Recall (Malignant): {recall_score(y_true_test, y_pred_test):.4f}")
print(f"Especificidad (Benign):            {tn_t / (tn_t + fp_t):.4f}")
print(f"Precisión (Malignant):             {precision_score(y_true_test, y_pred_test):.4f}")
print(f"F1-Score (Malignant):              {f1_score(y_true_test, y_pred_test):.4f}")
print(f"ROC-AUC (Test):                    {roc_auc_score(y_true_test, y_scores_test):.4f}")
print("=================================================================")

In [ ]:
from google.colab import files; files.download("resnet18_linea_base.pth")

In [ ]:
!pip install -q gradio==5.49.1 pydicom

In [ ]:
import app
app.demo.launch(share=True, debug=True)

In [ ]:
!pip install -q onnx onnxruntime

In [ ]:
!git clone https://github.com/McJulien/Mamo


In [ ]:
!pip install -q onnx onnxruntime
from google.colab import files
files.upload()

In [ ]:
# =============================================================================
# EJECUTAR EN COLAB (una sola celda), en la carpeta donde está resnet18_linea_base.pth
#   !pip install -q onnx onnxruntime
# Convierte el modelo a ONNX y comprueba que da lo mismo que PyTorch.
# El mapa Grad-CAM (clase "Maligno") queda calculado DENTRO del modelo:
# en ResNet18, Grad-CAM sobre layer4 == CAM = ReLU(sum_k w_k * A_k), por lo que
# no se necesitan gradientes en el servidor.
# =============================================================================
import numpy as np
import onnxruntime as ort
import torch
import torch.nn as nn
from torchvision import models

RUTA_PTH = "resnet18_linea_base.pth"
RUTA_ONNX = "resnet18_linea_base.onnx"

base = models.resnet18(weights=None)
base.fc = nn.Linear(base.fc.in_features, 2)
base.load_state_dict(torch.load(RUTA_PTH, map_location="cpu"))
base.eval()


class ConCAM(nn.Module):
    """Devuelve (logits, cam) para la clase 1 = Maligno."""
    def __init__(self, m):
        super().__init__()
        self.m = m

    def forward(self, x):
        m = self.m
        x = m.maxpool(m.relu(m.bn1(m.conv1(x))))
        a = m.layer4(m.layer3(m.layer2(m.layer1(x))))        # (1,512,7,7)
        logits = m.fc(torch.flatten(m.avgpool(a), 1))
        w = m.fc.weight[1].view(1, -1, 1, 1)
        cam = torch.relu((a * w).sum(dim=1))                   # (1,7,7)
        return logits, cam


envoltura = ConCAM(base).eval()
x = torch.randn(1, 3, 224, 224)
args = dict(input_names=["input"], output_names=["logits", "cam"], opset_version=17)
try:
    torch.onnx.export(envoltura, x, RUTA_ONNX, dynamo=False, **args)
except TypeError:  # versiones viejas de torch sin el argumento dynamo
    torch.onnx.export(envoltura, x, RUTA_ONNX, **args)

# ---------------- Verificación 1: logits ONNX == logits PyTorch
sess = ort.InferenceSession(RUTA_ONNX, providers=["CPUExecutionProvider"])
lo_onnx, cam_onnx = sess.run(None, {"input": x.numpy()})
with torch.no_grad():
    lo_torch = base(x).numpy()
print("Diferencia máx. en logits:", np.abs(lo_onnx - lo_torch).max())
assert np.allclose(lo_onnx, lo_torch, atol=1e-4), "¡Los logits no coinciden!"

# ---------------- Verificación 2: CAM == Grad-CAM real (con gradientes)
act, grd = {}, {}


def hook_layer4(modulo, entrada, salida):
    act["a"] = salida
    salida.register_hook(lambda g: grd.update(g=g))
    # (no se devuelve nada: devolver un valor reemplazaría la salida de layer4)


h = base.layer4.register_forward_hook(hook_layer4)
base.zero_grad()
base(x)[0, 1].backward()
h.remove()
gc = torch.relu((grd["g"].mean((2, 3), keepdim=True) * act["a"]).sum(1))[0].detach().numpy()
norm = lambda v: (v - v.min()) / (v.max() - v.min() + 1e-12)
print("Diferencia máx. CAM vs Grad-CAM (normalizados):", np.abs(norm(gc) - norm(cam_onnx[0])).max())

import os
print(f"OK -> {RUTA_ONNX} ({os.path.getsize(RUTA_ONNX)/1e6:.1f} MB). Descárgalo y súbelo a GitHub.")
from google.colab import files
files.download(RUTA_ONNX)